In [1]:
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv(r"E:\Data Science by SRK\Machine_learning\project\Medical Insurance Costs Dataset\cleaned_dataset_medical_cost_insurance.csv")

In [3]:
df

,age,gender,bmi,children,smoker,charges
0,19,0,27.900,0,1,16884.92400
1,18,1,33.770,1,0,1725.55230
2,28,1,33.000,3,0,4449.46200
3,33,1,22.705,0,0,21984.47061
4,32,1,28.880,0,0,3866.85520
...,...,...,...,...,...,...
1332,50,1,30.970,3,0,10600.54830
1333,18,0,31.920,0,0,2205.98080
1334,18,0,36.850,0,0,1629.83350
1335,21,0,25.800,0,0,2007.94500


In [4]:
df.shape

(1337, 6)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   gender    1337 non-null   int64  
 2   bmi       1337 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   int64  
 5   charges   1337 non-null   float64
dtypes: float64(2), int64(4)
memory usage: 62.8 KB


In [6]:
df.describe()

,age,gender,bmi,children,smoker,charges
count,1337.000000,1337.000000,1337.000000,1337.000000,1337.000000,1337.000000
mean,39.222139,0.504862,30.663452,1.095737,0.204936,13279.121487
std,14.044333,0.500163,6.100468,1.205571,0.403806,12110.359656
min,18.000000,0.000000,15.960000,0.000000,0.000000,1121.873900
25%,27.000000,0.000000,26.290000,0.000000,0.000000,4746.344000
50%,39.000000,1.000000,30.400000,1.000000,0.000000,9386.161300
75%,51.000000,1.000000,34.700000,2.000000,0.000000,16657.717450
max,64.000000,1.000000,53.130000,5.000000,1.000000,63770.428010


In [7]:
df.isnull().sum()

age         0
gender      0
bmi         0
children    0
smoker      0
charges     0
dtype: int64

# X & y

In [8]:
X = df.drop('charges', axis=1)
y = df['charges']

# Train test split

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state =9 )

# Modelling & Evaluation

**lasso regression with defaault parameters**

In [10]:
from sklearn.linear_model import ElasticNet
enr_base =  ElasticNet()
enr_base.fit(X_train,y_train)

# Prediction
train_predictions = enr_base.predict(X_train)
test_predictions = enr_base.predict(X_test)

# Evaluation
print("Train_R2", enr_base.score(X_train, y_train))

from sklearn.model_selection import cross_val_score
print("Cross_val_score", cross_val_score(enr_base, X, y, cv = 5).mean())

print("Test_R2", enr_base.score(X_test, y_test))

Train_R2 0.40398031945307333
Cross_val_score 0.3887213543302699
Test_R2 0.3684271884092043


**Appying Hyperparameter tuning for lasso regression**

In [11]:
from sklearn.model_selection import GridSearchCV

# model
estimator = ElasticNet()

# parameters & Values
param_grid = {"alpha" : [0.1,0.2,1,2,3,5,10], "l1_ratio" : [0.1,0.5,0.75,0.9,0.95,1]}

# Identifying the best value of the parameter within given values for the given data
model_hp = GridSearchCV(estimator, param_grid, cv= 5, scoring = 'neg_mean_squared_error')
model_hp.fit(X_train, y_train)
model_hp.best_params_

{'alpha': 10, 'l1_ratio': 1}

**Rebuilt lasso model using best hyperparameters**

In [12]:
# modelling

enr_best = ElasticNet(alpha=10, l1_ratio=1)
enr_best.fit(X_train,y_train)

print("Intercept : ", enr_best.intercept_)
print("coefficient : ", enr_best.coef_)
print("============================================")

# Predictions
train_predictions = enr_best.predict(X_train)
test_predictions  = enr_best.predict(X_test)

# Evaluation
print('Train_R2 : ', enr_best.score(X_train, y_train))
print("Test_R2 : ", enr_best.score(X_test, y_test))
print("cross_val_score : ", cross_val_score(enr_best, X, y, cv=5).mean())

Intercept :  -12141.353811913437
coefficient :  [  264.53924127   -77.07760056   318.0238622    405.21983259
 23925.6027292 ]
Train_R2 :  0.7593521947805205
Test_R2 :  0.700691116120383
cross_val_score :  0.7465805850880507
